Plot vector along Slice1 normal


In [ ]:
#### import the simple module from the paraview
from paraview.simple import *
#### disable automatic camera reset on 'Show'
paraview.simple._DisableFirstRenderCameraReset()
import os

src = r'..\recon-scripts\dmi_loss\16_layers_loss\results\loss1.6e+01_wphi1.0e+07\final.vtr'
base_output_path = os.environ.get('SLICE1_NORMAL_VECTOR_DIR', 'slice1_normal_reference_vector')
exist_ok = True
os.makedirs(base_output_path, exist_ok=True)

# Slice1/CutFunction/Plane in layers.pvsm
slice_normal = [0.5513973831191178, 0.8340427572064737, 0.0]
slice_origin = [-1.902839877015729e-8, -8.745272768155221e-9, 0.0]
slice_start = float(os.environ.get('SLICE1_NORMAL_START', '-6.3e-08'))
slice_end = float(os.environ.get('SLICE1_NORMAL_END', '6.3e-08'))
num_frames = int(os.environ.get('SLICE1_NORMAL_NUM_FRAMES', '272'))
vector_azimuth_degrees = 56.0
sample_rate = int(os.environ.get('SLICE1_NORMAL_SAMPLE_RATE', '6'))

def slice_center(distance):
    return [
        slice_normal[0] * distance,
        slice_normal[1] * distance,
        slice_origin[2],
    ]

def apply_vector_camera():
    renderView1.CameraPosition = [5.726384488316921e-07, -1.1810001865342201e-09, 0.0]
    renderView1.CameraFocalPoint = [-5.300009320308163e-10, -1.1810001865342201e-09, 0.0]
    renderView1.CameraViewUp = [0.0, 0.0, 1.0]
    renderView1.CameraParallelScale = 1.483469108507389e-07
    camera = GetActiveCamera()
    camera.Azimuth(vector_azimuth_degrees)

finalvtr = XMLRectilinearGridReader(FileName=[src])
finalvtr.PointArrayStatus = ['m', 'rgb']

renderView1 = GetActiveViewOrCreate('RenderView')
renderView1.ViewSize = [3000, 3000]

extractSubset1 = ExtractSubset(Input=finalvtr)
extractSubset1.VOI = [0, 135, 0, 135, 0, 110]
extractSubset1.SampleRateI = sample_rate
extractSubset1.SampleRateJ = sample_rate
extractSubset1.SampleRateK = sample_rate

clip1 = Clip(Input=extractSubset1)
clip1.ClipType = 'Cylinder'
clip1.ClipType.Axis = [0.0, 0.0, 1.0]
clip1.ClipType.Radius = 6.8e-08
clip1.Scalars = ['POINTS', '']

slice1 = Slice(Input=clip1)
slice1.SliceType = 'Plane'
slice1.SliceType.Normal = slice_normal
slice1.SliceType.Origin = slice_origin
slice1.HyperTreeGridSlicer = 'Plane'

pythonCalculator1 = PythonCalculator(Input=slice1)
pythonCalculator1.Expression = '-(arctan2(inputs[0].PointData["m"][:,1], inputs[0].PointData["m"][:,0]) + 2*np.pi) % (2*np.pi)'
pythonCalculator1.ArrayName = 'xy'

calculator1 = Calculator(Input=pythonCalculator1)
calculator1.ResultArrayName = 'z'
calculator1.Function = 'm_Z'
zLUT = GetColorTransferFunction('z')
zPWF = GetOpacityTransferFunction('z')

glyph1 = Glyph(Input=pythonCalculator1, GlyphType='Arrow')
glyph1.OrientationArray = ['POINTS', 'm']
glyph1.ScaleArray = ['POINTS', 'No scale array']
glyph1.ScaleFactor = 1.385020453881225e-08
glyph1.GlyphTransform = 'Transform2'
glyph1.ScaleFactor = 8e-09
glyph1.GlyphMode = 'All Points'
glyph1.GlyphTransform.Translate = [-0.5, 0.0, 0.0]
glyph1Display = Show(glyph1, renderView1, 'GeometryRepresentation')
glyph1Display.Representation = 'Surface'
glyph1Display.ColorArrayName = [None, '']
glyph1Display.OSPRayScaleArray = 'rgb'
glyph1Display.OSPRayScaleFunction = 'PiecewiseFunction'
glyph1Display.SelectOrientationVectors = 'None'
glyph1Display.ScaleFactor = 1.4473233278522457e-08
glyph1Display.SelectScaleArray = 'None'
glyph1Display.GlyphType = 'Arrow'
glyph1Display.GlyphTableIndexArray = 'None'
glyph1Display.GaussianRadius = 7.236616639261228e-10
glyph1Display.SetScaleArray = ['POINTS', 'rgb']
glyph1Display.ScaleTransferFunction = 'PiecewiseFunction'
glyph1Display.OpacityArray = ['POINTS', 'rgb']
glyph1Display.OpacityTransferFunction = 'PiecewiseFunction'
glyph1Display.DataAxesGrid = 'GridAxesRepresentation'
glyph1Display.PolarAxes = 'PolarAxesRepresentation'
glyph1Display.ScaleTransferFunction.Points = [-0.9030365018296727, 0.0, 0.5, 0.0, 0.6671053584747355, 1.0, 0.5, 0.0]
glyph1Display.OpacityTransferFunction.Points = [-0.9030365018296727, 0.0, 0.5, 0.0, 0.6671053584747355, 1.0, 0.5, 0.0]
glyph1Display.SetScalarBarVisibility(renderView1, False)

ColorBy(glyph1Display, ('POINTS', 'xy'))
HideScalarBarIfNotNeeded(zLUT, renderView1)
glyph1Display.RescaleTransferFunctionToDataRange(True, False)
xyLUT = GetColorTransferFunction('xy')
xyPWF = GetOpacityTransferFunction('xy')
glyph1Display.InterpolateScalarsBeforeMapping = 0
xyLUT.ApplyPreset('hsv', True)
glyph1.GlyphType.TipResolution = 8
glyph1.GlyphType.TipRadius = 0.15
glyph1.GlyphType.TipLength = 0.5
glyph1.GlyphType.ShaftResolution = 6
glyph1.GlyphType.ShaftRadius = 0.06
xyLUT.RescaleTransferFunction(2.951596406219892e-05, 6.2830244480651665)
xyPWF.RescaleTransferFunction(2.951596406219892e-05, 6.2830244480651665)

glyph2 = Glyph(Input=calculator1, GlyphType='Arrow')
glyph2.OrientationArray = ['POINTS', 'm']
glyph2.ScaleArray = ['POINTS', 'z']
glyph2.ScaleFactor = 1.4039279250255278e-08
glyph2.GlyphTransform = 'Transform2'
glyph2.ScaleArray = ['POINTS', 'No scale array']
glyph2.ScaleFactor = 8e-09
glyph2.GlyphMode = 'All Points'
glyph2.GlyphTransform.Translate = [-0.5, 0.0, 0.0]
glyph2Display = Show(glyph2, renderView1, 'GeometryRepresentation')
glyph2Display.Representation = 'Surface'
glyph2Display.ColorArrayName = ['POINTS', 'z']
glyph2Display.LookupTable = zLUT
glyph2Display.OSPRayScaleArray = 'z'
glyph2Display.OSPRayScaleFunction = 'PiecewiseFunction'
glyph2Display.SelectOrientationVectors = 'None'
glyph2Display.ScaleFactor = 1.4579482865428873e-08
glyph2Display.SelectScaleArray = 'z'
glyph2Display.GlyphType = 'Arrow'
glyph2Display.GlyphTableIndexArray = 'z'
glyph2Display.GaussianRadius = 7.289741432714436e-10
glyph2Display.SetScaleArray = ['POINTS', 'z']
glyph2Display.ScaleTransferFunction = 'PiecewiseFunction'
glyph2Display.OpacityArray = ['POINTS', 'z']
glyph2Display.OpacityTransferFunction = 'PiecewiseFunction'
glyph2Display.DataAxesGrid = 'GridAxesRepresentation'
glyph2Display.PolarAxes = 'PolarAxesRepresentation'
glyph2Display.ScaleTransferFunction.Points = [-0.9030365018296727, 0.0, 0.5, 0.0, 0.6671053584747355, 1.0, 0.5, 0.0]
glyph2Display.OpacityTransferFunction.Points = [-0.9030365018296727, 0.0, 0.5, 0.0, 0.6671053584747355, 1.0, 0.5, 0.0]
glyph2Display.SetScalarBarVisibility(renderView1, False)
zLUT.ApplyPreset('X Ray', True)
zPWF.Points = [-0.9030364751815796, 1.0, 0.5, 0.0, -0.13215357065200806, 0.0, 0.5, 0.0, 0.6671053584747355, 1.0, 0.5, 0.0]
zLUT.RGBPoints = [-0.9030365018296727, 1.0, 1.0, 1.0, -0.1652591, 1.0, 1.0, 1.0, -0.0848601, 0.0, 0.0, 0.0, 0.6671053584747355, 0.0, 0.0, 0.0]
zLUT.EnableOpacityMapping = 1
glyph2.GlyphType.TipResolution = 8
glyph2.GlyphType.TipRadius = 0.15
glyph2.GlyphType.TipLength = 0.5
glyph2.GlyphType.ShaftResolution = 6
glyph2.GlyphType.ShaftRadius = 0.06

# side view yz, then azimuth rotation requested by user
apply_vector_camera()

Hide(finalvtr, renderView1)
Hide(extractSubset1, renderView1)
Hide(clip1, renderView1)
Hide(slice1, renderView1)
Hide(pythonCalculator1, renderView1)
Hide(calculator1, renderView1)

for layer in range(num_frames):
    if num_frames == 1:
        distance = slice_start
    else:
        distance = slice_start + (slice_end - slice_start) * layer / (num_frames - 1)

    slice1.SliceType.Origin = slice_center(distance)
    renderView1.Update()
    if layer == 4:
        renderView1.ResetCamera()
        apply_vector_camera()

    des = os.path.join(base_output_path, f'{layer}.png')
    print(f"Saving layer {layer} to {des}...")

    SaveScreenshot(des, renderView1, ImageResolution=[1200, 1192],
        OverrideColorPalette='WhiteBackground',
        CompressionLevel='0')
print("Done!")


Plot cross section along Slice1 normal


In [ ]:
#### import the simple module from the paraview
from paraview.simple import *
#### disable automatic camera reset on 'Show'
paraview.simple._DisableFirstRenderCameraReset()
import os

src = r'..\recon-scripts\dmi_loss\16_layers_loss\results\loss1.6e+01_wphi1.0e+07\final.vtr'
base_output_path = os.environ.get('SLICE1_NORMAL_CROSS_DIR', 'slice1_normal_reference_cross')
exist_ok = True
os.makedirs(base_output_path, exist_ok=True)

# Slice1/CutFunction/Plane in layers.pvsm
slice_normal = [0.5513973831191178, 0.8340427572064737, 0.0]
slice_origin = [-1.902839877015729e-8, -8.745272768155221e-9, 0.0]
slice_tangent = [-slice_normal[1], slice_normal[0], 0.0]
slice_z = [0.0, 0.0, 1.0]
slice_start = float(os.environ.get('SLICE1_NORMAL_START', '-6.3e-08'))
slice_end = float(os.environ.get('SLICE1_NORMAL_END', '6.3e-08'))
num_frames = int(os.environ.get('SLICE1_NORMAL_NUM_FRAMES', '272'))

def set_if_exists(proxy, name, value):
    try:
        setattr(proxy, name, value)
    except Exception:
        pass

def plane_points(distance):
    center = [
        slice_normal[0] * distance,
        slice_normal[1] * distance,
        slice_origin[2],
    ]
    half_w = 6.8e-08
    half_h = 5.83e-08
    origin = [
        center[0] - slice_tangent[0] * half_w - slice_z[0] * half_h,
        center[1] - slice_tangent[1] * half_w - slice_z[1] * half_h,
        center[2] - slice_tangent[2] * half_w - slice_z[2] * half_h,
    ]
    point1 = [
        center[0] + slice_tangent[0] * half_w - slice_z[0] * half_h,
        center[1] + slice_tangent[1] * half_w - slice_z[1] * half_h,
        center[2] + slice_tangent[2] * half_w - slice_z[2] * half_h,
    ]
    point2 = [
        center[0] - slice_tangent[0] * half_w + slice_z[0] * half_h,
        center[1] - slice_tangent[1] * half_w + slice_z[1] * half_h,
        center[2] - slice_tangent[2] * half_w + slice_z[2] * half_h,
    ]
    return center, origin, point1, point2

finalvtr = XMLRectilinearGridReader(FileName=[src])
finalvtr.PointArrayStatus = ['m', 'rgb']

renderView1 = GetActiveViewOrCreate('RenderView')
renderView1.ViewSize = [3000, 3000]

extractSubset1 = ExtractSubset(Input=finalvtr)
extractSubset1.VOI = [0, 135, 0, 135, 0, 110]

clip1 = Clip(Input=extractSubset1)
clip1.ClipType = 'Cylinder'
clip1.ClipType.Axis = [0.0, 0.0, 1.0]
clip1.ClipType.Radius = 6.8e-08
clip1.Scalars = ['POINTS', '']

pythonCalculator1 = PythonCalculator(Input=clip1)
pythonCalculator1.Expression = '-(arctan2(inputs[0].PointData["m"][:,1], inputs[0].PointData["m"][:,0]) + 2*np.pi) % (2*np.pi)'
pythonCalculator1.ArrayName = 'xy'
pythonCalculator1Display = Show(pythonCalculator1, renderView1, 'UnstructuredGridRepresentation')
pythonCalculator1Display.Representation = 'Surface'
pythonCalculator1Display.ColorArrayName = [None, '']
pythonCalculator1Display.OSPRayScaleArray = 'm'
pythonCalculator1Display.OSPRayScaleFunction = 'PiecewiseFunction'
pythonCalculator1Display.SelectOrientationVectors = 'm'
pythonCalculator1Display.ScaleFactor = 1.4039279250255278e-08
pythonCalculator1Display.SelectScaleArray = 'None'
pythonCalculator1Display.GlyphType = 'Arrow'
pythonCalculator1Display.GlyphTableIndexArray = 'None'
pythonCalculator1Display.GaussianRadius = 7.019639625127639e-10
pythonCalculator1Display.SetScaleArray = ['POINTS', 'm']
pythonCalculator1Display.ScaleTransferFunction = 'PiecewiseFunction'
pythonCalculator1Display.OpacityArray = ['POINTS', 'm']
pythonCalculator1Display.OpacityTransferFunction = 'PiecewiseFunction'
pythonCalculator1Display.DataAxesGrid = 'GridAxesRepresentation'
pythonCalculator1Display.PolarAxes = 'PolarAxesRepresentation'
set_if_exists(pythonCalculator1Display, 'ScalarOpacityUnitDistance', 1.984227587500313e-09)

pythonCalculator1Display.ScaleTransferFunction.Points = [-0.9999826550483704, 0.0, 0.5, 0.0, 0.9999998807907104, 1.0, 0.5, 0.0]
pythonCalculator1Display.OpacityTransferFunction.Points = [-0.9999826550483704, 0.0, 0.5, 0.0, 0.9999998807907104, 1.0, 0.5, 0.0]
Hide(clip1, renderView1)
renderView1.Update()
pythonCalculator1Display.InterpolateScalarsBeforeMapping = 0
ColorBy(pythonCalculator1Display, ('POINTS', 'xy'))
pythonCalculator1Display.RescaleTransferFunctionToDataRange(True, False)
pythonCalculator1Display.SetScalarBarVisibility(renderView1, False)
xyLUT = GetColorTransferFunction('xy')
xyPWF = GetOpacityTransferFunction('xy')
xyLUT.ApplyPreset('hsv', True)

# 计算 Z 分量 (Calculator)
calculator1 = Calculator(Input=pythonCalculator1)
calculator1.ResultArrayName = 'z'
calculator1.Function = 'm_Z'

calculator1Display = Show(calculator1, renderView1, 'UnstructuredGridRepresentation')
zLUT = GetColorTransferFunction('z')
zPWF = GetOpacityTransferFunction('z')
calculator1Display.Representation = 'Surface'
calculator1Display.ColorArrayName = ['POINTS', 'z']
calculator1Display.LookupTable = zLUT
calculator1Display.OSPRayScaleArray = 'z'
calculator1Display.OSPRayScaleFunction = 'PiecewiseFunction'
calculator1Display.SelectOrientationVectors = 'm'
calculator1Display.ScaleFactor = 1.4039279250255278e-08
calculator1Display.SelectScaleArray = 'z'
calculator1Display.GlyphType = 'Arrow'
calculator1Display.GlyphTableIndexArray = 'z'
calculator1Display.GaussianRadius = 7.019639625127639e-10
calculator1Display.SetScaleArray = ['POINTS', 'z']
calculator1Display.ScaleTransferFunction = 'PiecewiseFunction'
calculator1Display.OpacityArray = ['POINTS', 'z']
calculator1Display.OpacityTransferFunction = 'PiecewiseFunction'
calculator1Display.DataAxesGrid = 'GridAxesRepresentation'
calculator1Display.PolarAxes = 'PolarAxesRepresentation'
set_if_exists(calculator1Display, 'ScalarOpacityFunction', zPWF)
set_if_exists(calculator1Display, 'ScalarOpacityUnitDistance', 1.984227587500313e-09)

calculator1Display.ScaleTransferFunction.Points = [-0.9999999403953552, 0.0, 0.5, 0.0, 0.9999898076057434, 1.0, 0.5, 0.0]
calculator1Display.OpacityTransferFunction.Points = [-0.9999999403953552, 0.0, 0.5, 0.0, 0.9999898076057434, 1.0, 0.5, 0.0]
renderView1.Update()
zLUT.ApplyPreset('X Ray', True)
zPWF.Points = [-0.9999999403953552, 1.0, 0.5, 0.0, 0.012043051421642303, 0.0, 0.5, 0.0, 0.9999898076057434, 1.0, 0.5, 0.0]
zLUT.RGBPoints = [-0.9999999403953552, 1.0, 1.0, 1.0, -0.03614947199821472, 1.0, 1.0, 1.0, 0.030115246772766113, 0.0, 0.0, 0.0, 0.9999898076057434, 0.0, 0.0, 0.0]
zLUT.EnableOpacityMapping = 1

SetActiveSource(pythonCalculator1)
pythonCalculator1Display = Show(pythonCalculator1, renderView1, 'UnstructuredGridRepresentation')
pythonCalculator1Display.SetScalarBarVisibility(renderView1, True)
Hide(extractSubset1, renderView1)
Hide(finalvtr, renderView1)
pythonCalculator1Display.SetScalarBarVisibility(renderView1, False)
SetActiveSource(calculator1)
calculator1Display.SetScalarBarVisibility(renderView1, False)

plane1 = Plane()
plane1.XResolution = 1
plane1.YResolution = 1
_, plane_origin, plane_point1, plane_point2 = plane_points(slice_start)
plane1.Origin = plane_origin
plane1.Point1 = plane_point1
plane1.Point2 = plane_point2

calculator2 = Calculator(Input=plane1)
calculator2.ResultArrayName = '1'
calculator2.Function = '1'
calculator2Display = Show(calculator2, renderView1, 'GeometryRepresentation')
calculator2Display.Representation = 'Surface'
calculator2Display.Opacity = 0.5
ColorBy(calculator2Display, ('POINTS', '1'))

renderView1.ResetCamera()
renderView1.CameraPosition = [3.902963157555595e-07, -5.300000000000054e-10, 2.2533768267345233e-07]
renderView1.CameraFocalPoint = [-4.5899333767903354e-10, -5.300000000000054e-10, -2.6500021880430646e-10]
renderView1.CameraViewUp = [-0.49999999999999994, 0.0, 0.8660254037844387]
renderView1.CameraParallelScale = 1.1678054191748132e-07

for layer in range(num_frames):
    if num_frames == 1:
        distance = slice_start
    else:
        distance = slice_start + (slice_end - slice_start) * layer / (num_frames - 1)

    _, plane_origin, plane_point1, plane_point2 = plane_points(distance)
    plane1.Origin = plane_origin
    plane1.Point1 = plane_point1
    plane1.Point2 = plane_point2
    renderView1.Update()
    des = os.path.join(base_output_path, f'{layer}.png')
    print(f"Saving layer {layer} to {des}...")

    SaveScreenshot(des, renderView1, ImageResolution=[1200, 1192],
        OverrideColorPalette='WhiteBackground',
        CompressionLevel='0')
print("Done!")


Synthetic video of Slice1-normal cross section


In [ ]:
import os
import glob
import re
import subprocess
from PIL import Image
import numpy as np
import imageio.v2 as imageio

DIR_A = 'slice1_normal_reference_cross'
DIR_B = 'slice1_normal_reference_vector'
COMPOSITE_DIR = 'slice1_normal_reference_frames'
OUTPUT_VIDEO_FILE = 'slice1_normal_reference_video.mp4'
SOURCE_PDF_PATH = 'coordinate.pdf'
TEMP_PNG_PATH = 'temp_overlay_transparent.png'
PDF_ZOOM = 4

FPS = 10
SECONDS_PER_IMAGE = 1/FPS

os.makedirs(COMPOSITE_DIR, exist_ok=True)

def make_transparent(png_path):
    try:
        img = Image.open(png_path).convert("RGBA")
        data = np.array(img)

        red, green, blue, alpha = data.T
        white_areas = (red > 240) & (green > 240) & (blue > 240)
        data[..., 3][white_areas.T] = 0

        img_new = Image.fromarray(data)
        img_new.save(png_path)
        return True
    except Exception as e:
        print(f"  -> error: {e}")
        return False

def convert_pdf_to_transparent_png(pdf_path, output_png_path, zoom=4):
    if not os.path.exists(pdf_path):
        print(f"error: cannot found pdf file: {pdf_path}")
        return False

    try:
        import fitz
        doc = fitz.open(pdf_path)
        page = doc.load_page(0)

        mat = fitz.Matrix(zoom, zoom)
        pix = page.get_pixmap(matrix=mat, alpha=True)

        pix.save(output_png_path)
        doc.close()

        success = make_transparent(output_png_path)
        return success
    except ModuleNotFoundError:
        bundled_python = sys.executable
        if not os.path.exists(bundled_python):
            print("There was an error in the PDF conversion process: fitz is unavailable.")
            return False
        helper = r'''
import sys
import fitz
import numpy as np
from PIL import Image

pdf_path, output_png_path, zoom_value = sys.argv[1], sys.argv[2], float(sys.argv[3])
doc = fitz.open(pdf_path)
page = doc.load_page(0)
pix = page.get_pixmap(matrix=fitz.Matrix(zoom_value, zoom_value), alpha=True)
pix.save(output_png_path)
doc.close()

img = Image.open(output_png_path).convert("RGBA")
data = np.array(img)
red, green, blue, alpha = data.T
white_areas = (red > 240) & (green > 240) & (blue > 240)
data[..., 3][white_areas.T] = 0
Image.fromarray(data).save(output_png_path)
'''
        result = subprocess.run(
            [bundled_python, '-c', helper, pdf_path, output_png_path, str(zoom)],
            text=True,
            capture_output=True,
        )
        if result.returncode != 0:
            print(f"There was an error in the PDF conversion process: {result.stderr.strip()}")
            return False
        return True
    except Exception as e:
        print(f"There was an error in the PDF conversion process: {e}")
        return False

def get_sorted_common_files(dir_a, dir_b):
    try:
        files_a = os.listdir(dir_a)
        files_b_set = set(os.listdir(dir_b))
    except FileNotFoundError as e:
        print(f"error: not found dir {e.filename}.")
        return []

    common_files = []
    image_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff')

    for f in files_a:
        if f in files_b_set and f.lower().endswith(image_extensions):
            common_files.append(f)

    if not common_files:
        print("warning: not found figure files between a and b.")
        return []

    def sort_key(filename):
        try:
            name_without_ext = os.path.splitext(filename)[0]
            return int(name_without_ext)
        except ValueError:
            print(f"warning: The file name '{filename}' cannot be parsed by numbers.")
            return 0

    common_files.sort(key=sort_key)

    print(f"--- Find and sorted {len(common_files)} files ---")
    return common_files

def stitch_images(img_a, img_b, overlay_img=None):
    width_a, height_a = img_a.size
    width_b, height_b = img_b.size

    total_width = width_a + width_b
    max_height = max(height_a, height_b)
    stitched_img = Image.new('RGB', (total_width, max_height), (0, 0, 0))
    stitched_img.paste(img_a, (0, (max_height - height_a) // 2))
    stitched_img.paste(img_b, (width_a, (max_height - height_b) // 2))

    if overlay_img:
        ov_width, ov_height = overlay_img.size

        x_pos = width_a - (ov_width // 2)
        y_pos = 20

        if overlay_img.mode == 'RGBA':
            stitched_img.paste(overlay_img, (x_pos, y_pos), mask=overlay_img)
        else:
            stitched_img.paste(overlay_img, (x_pos, y_pos))

    return stitched_img

def create_video(image_filenames, overlay_path):
    print("start creating the video...")

    overlay_img = None
    if os.path.exists(overlay_path):
        try:
            overlay_img = Image.open(overlay_path)
            print("Overlay image loaded successfully.")
        except Exception as e:
            print(f"Error loading overlay image: {e}")
    else:
        print("Warning: Overlay image file not found, video will have no icon.")

    repeat_frames = max(1, int(FPS * SECONDS_PER_IMAGE))

    with imageio.get_writer(OUTPUT_VIDEO_FILE, fps=FPS, macro_block_size=1) as writer:
        for idx, filename in enumerate(image_filenames):
            path_a = os.path.join(DIR_A, filename)
            path_b = os.path.join(DIR_B, filename)

            if not os.path.exists(path_a) or not os.path.exists(path_b):
                print(f"warning: unable to find {path_a} or {path_b}, skipping this frame.")
                continue

            img_a = Image.open(path_a).convert('RGB')
            img_b = Image.open(path_b).convert('RGB')
            stitched_frame = stitch_images(img_a, img_b, overlay_img)

            composite_path = os.path.join(COMPOSITE_DIR, filename)
            stitched_frame.save(composite_path)

            for _ in range(repeat_frames):
                writer.append_data(imageio.imread(composite_path))

            if idx % 10 == 0:
                print(f"processed: {idx}/{len(image_filenames)} figures")

    print(f"Video saved to: {OUTPUT_VIDEO_FILE}")

if __name__ == "__main__":
    for old_frame in glob.glob(os.path.join(COMPOSITE_DIR, "*.png")):
        os.remove(old_frame)

    convert_success = convert_pdf_to_transparent_png(SOURCE_PDF_PATH, TEMP_PNG_PATH, PDF_ZOOM)
    if not convert_success:
        print("\nWarning: PDF conversion failed.")

    files_to_process = get_sorted_common_files(DIR_A, DIR_B)
    if files_to_process:
        create_video(files_to_process, TEMP_PNG_PATH)
    else:
        print("Error: No matching file pairs found.")

    if os.path.exists(TEMP_PNG_PATH):
        os.remove(TEMP_PNG_PATH)
